In [1]:
pip install pyngrok

In [2]:
pip install -U bitsandbytes

In [3]:
pip install langchain-community

In [4]:
pip install supabase

In [5]:
pip install faiss-cpu

In [16]:
import nest_asyncio
import uvicorn
import torch
import gc
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from huggingface_hub import snapshot_download, login
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from supabase import create_client, Client
from typing import Optional, List

nest_asyncio.apply()

# Configuration and Component Loading
MODEL_REPO_ID = "Wiefdw/merged-tax-raft-mistral-7b"
VECTOR_REPO_ID = "Wiefdw/tax-indonesia-vectordb"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LOCAL_DB_PATH = "./vector_db"

# Credentials Configuration
HF_TOKEN = "YOUR_HUGGING_FACE_TOKEN"
login(token=HF_TOKEN)

SUPABASE_URL = "https://nzteayiwvcrhpdjfpcic.supabase.co"
SUPABASE_KEY = "YOUR_SUPABASE_KEY"

# Supabase Client and FastAPI App Initialization
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
app = FastAPI(title="API Chatbot Pajak (Mistral)")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

# Model and Database Loading Process
print("Downloading and loading all AI components...")
snapshot_download(repo_id=VECTOR_REPO_ID, repo_type="dataset", local_dir=LOCAL_DB_PATH, token=HF_TOKEN)
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME, model_kwargs={'device': 'cpu'})
vectorstore = FAISS.load_local(LOCAL_DB_PATH, embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, llm_int8_enable_fp32_cpu_offload=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_REPO_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16, token=HF_TOKEN, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO_ID, token=HF_TOKEN, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
llm_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=1024)
print("All AI components loaded successfully.")

# API Endpoint Definitions

# Pydantic Models
class LoginRequest(BaseModel): email: str; password: str
class RegisterRequest(BaseModel): email: str; password: str
class AskRequest(BaseModel): question: str; conversation_id: Optional[str] = None
class FeedbackRequest(BaseModel): message_id: str; rating: int

# Dependency Function for Authentication
async def get_current_user(request: Request) -> dict:
    auth_header = request.headers.get("Authorization")
    if not auth_header or not auth_header.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing or invalid authorization header")
    token = auth_header.split(" ")[1]
    try:
        user_data = supabase.auth.get_user(token)
        return user_data.user
    except Exception:
        raise HTTPException(status_code=401, detail="Invalid token")

async def get_optional_current_user(request: Request) -> Optional[dict]:
    auth_header = request.headers.get("Authorization")
    if not auth_header or not auth_header.startswith("Bearer "):
        return None
    token = auth_header.split(" ")[1]
    try:
        user_data = supabase.auth.get_user(token)
        return user_data.user
    except Exception:
        return None

# Endpoints
@app.post("/register", tags=["Authentication"])
async def register(req: RegisterRequest):
    try:
        res = supabase.auth.sign_up({"email": req.email, "password": req.password})
        return {"message": "Registration successful, please check email for verification."}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/login", tags=["Authentication"])
async def login(req: LoginRequest):
    try:
        res = supabase.auth.sign_in_with_password({"email": req.email, "password": req.password})
        return {"access_token": res.session.access_token, "token_type": "bearer"}
    except Exception:
        raise HTTPException(status_code=401, detail="Incorrect email or password")

@app.get("/conversations", tags=["Chat"])
async def get_conversations(user: dict = Depends(get_current_user)):
    res = supabase.table("conversations").select("id, title, created_at").eq("user_id", user.id).order("created_at", desc=True).execute()
    return res.data

@app.get("/conversations/{conversation_id}", tags=["Chat"])
async def get_messages(conversation_id: str, user: dict = Depends(get_current_user)):
    res = supabase.table("messages").select("*", "feedback(*)").eq("conversation_id", conversation_id).eq("user_id", user.id).order("created_at").execute()
    return res.data

@app.delete("/conversations/{conversation_id}", tags=["Chat"])
async def delete_conversation(conversation_id: str, user: dict = Depends(get_current_user)):
    supabase.table("conversations").delete().eq("id", conversation_id).eq("user_id", user.id).execute()
    return {"message": "Conversation deleted successfully"}

@app.post("/ask", tags=["Chat"])
async def ask(req: AskRequest, user: Optional[dict] = Depends(get_optional_current_user)):
    question = req.question
    conversation_id = req.conversation_id

    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    # Prompt format for Mistral model
    prompt = f"""<s>[INST]
Jawab pertanyaan berikut HANYA dalam Bahasa Indonesia, berdasarkan konteks yang diberikan.

Fokuskan jawaban hanya pada POIN-POIN UTAMA yang relevan.
Jangan bertele-tele.
Gunakan gaya jawab yang padat, jelas, dan langsung ke inti.

Pertanyaan: {question}

Konteks:
{context}
[/INST]"""

    response = llm_pipeline(prompt)

    # Parse Mistral model's response
    raw_answer = response[0]["generated_text"]
    answer = raw_answer.split("[/INST]")[-1].strip().replace("</s>", "")

    if not answer:
        answer = "Maaf, model tidak memberikan jawaban yang valid. Silakan coba ajukan pertanyaan yang berbeda."

    # Database storage logic (only if user is logged in)
    if user:
        user_id = user.id
        if not conversation_id:
            title = question[:50]
            new_convo = supabase.table("conversations").insert({"user_id": user_id, "title": title}).execute().data[0]
            conversation_id = new_convo['id']

        messages_to_insert = [
            {"user_id": user_id, "conversation_id": conversation_id, "role": "user", "content": question},
            {"user_id": user_id, "conversation_id": conversation_id, "role": "assistant", "content": answer}
        ]
        message_data = supabase.table("messages").insert(messages_to_insert).execute().data
        assistant_message_id = next((msg['id'] for msg in message_data if msg['role'] == 'assistant'), None)
        return {"answer": answer, "conversation_id": conversation_id, "message_id": assistant_message_id}
    else:
        return {"answer": answer, "conversation_id": None, "message_id": None}

@app.post("/feedback", tags=["Feedback"])
async def record_feedback(req: FeedbackRequest, user: dict = Depends(get_current_user)):
    supabase.table("feedback").upsert({
        "message_id": req.message_id,
        "user_id": user.id,
        "rating": req.rating
    }, on_conflict="message_id, user_id").execute()
    return {"message": "Feedback recorded successfully"}

# Run Server and Ngrok
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")
print("Starting server and opening Ngrok connection on port 7860...")
public_url = ngrok.connect(7860)
print('Public URL:', public_url)

import threading
import time

def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=7860, loop="asyncio")
    server = uvicorn.Server(config)
    server.run()

# Start the server in a new thread
thread = threading.Thread(target=run_server)
thread.start()

# Give the server a moment to start up
time.sleep(5)

print("FastAPI server should be running in a background thread.")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
/usr/lib/python3.12/copy.py:252: RuntimeWarning: coroutine 'Server.serve' was never awaited
  args = (deepcopy(arg, memo) for arg in args)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

All AI components loaded successfully.
Starting server and opening Ngrok connection on port 7860...
Public URL: NgrokTunnel: "https://1cbd-34-125-195-200.ngrok-free.app" -> "http://localhost:7860"


INFO:     Started server process [6289]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)


FastAPI server should be running in a background thread.
